<a href="https://colab.research.google.com/github/kihahu/kikuyu-tts/blob/initial-import/notebooks/train_kikuyu_vits_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train Kikuyu VITS From Scratch (WaxalNLP `kik_tts`)

This notebook executes the full pipeline in Colab:
1. Environment setup
2. Dataset preprocessing + speaker-disjoint manifests
3. Tokenizer/vocab build
4. From-scratch training with resume-safe checkpointing
5. Evaluation + checkpoint selection
6. Local integration artifact packaging


In [22]:
!pip install -U pip setuptools wheel
!pip install datasets[audio] soundfile librosa pyyaml huggingface_hub
# Installing TTS directly from GitHub with build dependencies pre-installed
!pip install coqui-tts

In [23]:
from google.colab import drive
drive.mount('/kikuyu-tts')


Drive already mounted at /kikuyu-tts; to attempt to forcibly remount, call drive.mount("/kikuyu-tts", force_remount=True).


In [24]:
%cd /content/kikuyu-tts
!git pull
#%cd /content/
#!git clone https://github.com/kihahu/kikuyu-tts.git
#%cd /content/


/content/kikuyu-tts
Already up to date.


In [25]:
!python scripts/prepare_waxal_kik_tts.py \
  --dataset-name google/WaxalNLP \
  --dataset-config kik_tts \
  --split train \
  --output-dir data/waxal_kik_tts \
  --target-sample-rate 16000 \
  --min-duration-sec 0.6 \
  --max-duration-sec 25.0 \
  --min-rms 0.0035 \
  --seed 42 \
  --dev-ratio 0.10 \
  --test-ratio 0.05


Resolving data files: 100% 72/72 [00:00<00:00, 12067.53it/s]
{
  "config": {
    "dataset_name": "google/WaxalNLP",
    "dataset_config": "kik_tts",
    "split": "train",
    "target_sample_rate": 16000,
    "min_duration_sec": 0.6,
    "max_duration_sec": 25.0,
    "min_rms": 0.0035,
    "seed": 42,
    "dev_ratio": 0.1,
    "test_ratio": 0.05
  },
  "total_rows_after_filter": 1170,
  "split_sizes": {
    "train": 857,
    "dev": 153,
    "test": 160
  },
  "unique_speakers": 8,
  "skip_reasons": {
    "too_long": 432
  }
}


In [26]:
!python scripts/build_kikuyu_vocab.py \
  --train-manifest data/waxal_kik_tts/manifests/train.jsonl \
  --dev-manifest data/waxal_kik_tts/manifests/dev.jsonl \
  --out-dir artifacts/tokenizer_kikuyu_char


Wrote vocab with 69 tokens to artifacts/tokenizer_kikuyu_char


In [5]:
!pip uninstall TTS trainer coqpit
!pip cache purge
!pip install coqui-tts

#%cd /content
%cd /content/kikuyu-tts
!git pull
!pip install -e .
%cd /content/kikuyu-tts

Files removed: 18 (1.4 MB)
Directories removed: 0
/content/kikuyu-tts
Already up to date.
Obtaining file:///content/kikuyu-tts
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for kikuyu-tts (pyproject.toml) ... done
  Created wheel for kikuyu-tts: filename=kikuyu_tts-0.1.0-0.editable-py3-none-any.whl size=1448 sha256=5debd6609a0bed92a5e472c8006d1404034a168fa53ecdb9711ce9f51e07df6f
  Stored in directory: /tmp/pip-ephem-wheel-cache-vu5yped7/wheels/c9/e6/d8/8ef5d8203dd225906790609ab80b2569b43b884d17d10a5843
Successfully built kikuyu-tts
  Attempting uninstall: kikuyu-tts
    Found existing installation: kikuyu-tts 0.1.0
    Uninstalling kikuyu-tts-0.1.0:
      Successfully uninstalled kikuyu-tts-0.1.0
/content/kikuyu-tts


In [8]:
# Start fresh
!python -m TTS.bin.train_tts --config_path /content/kikuyu-tts/artifacts/colab_runs/kikuyu_vits_scratch/coqui_vits_config.yaml



/usr/local/lib/python3.12/dist-packages/coqpit/coqpit.py:875: UserWarning: Type mismatch in VitsConfig
Failed to deserialize field: grad_clip (list[float]) = 1.0
Value `1.0` does not match field type `list[float]`
Replaced it with field's default value: [1000, 1000]
  self.deserialize(data)
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/TTS/bin/train_tts.py", line 77, in <module>
    main()
  File "/usr/local/lib/python3.12/dist-packages/TTS/bin/train_tts.py", line 52, in main
    train_samples, eval_samples = load_tts_samples(
                                  ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/TTS/tts/datasets/__init__.py", line 126, in load_tts_samples
    meta_data_train = formatter(root_path, meta_file_train, ignored_speakers=ignored_speakers)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [29]:
# Resume mode
!python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/kikuyu-tts/ \
  --resume


Traceback (most recent call last):
  File "/content/kikuyu-tts/scripts/colab_train_vits_scratch.py", line 157, in <module>
    main()
  File "/content/kikuyu-tts/scripts/colab_train_vits_scratch.py", line 131, in main
    resume_ckpt = merge_resume_checkpoint(local_output_dir, drive_output_dir)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/kikuyu-tts/scripts/colab_train_vits_scratch.py", line 31, in merge_resume_checkpoint
    if not drive_output_dir.exists():
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 860, in exists
    self.stat(follow_symlinks=follow_symlinks)
  File "/usr/lib/python3.12/pathlib.py", line 840, in stat
    return os.stat(self, follow_symlinks=follow_symlinks)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected: '/content/drive/MyDrive/kikuyu-tts/checkpoints/kikuyu_vits_scratch'


In [30]:
# Optional: push checkpoints to HF Hub
!huggingface-cli login
!python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/kikuyu-tts/ \
  --resume \
  --push-hf




  A new version of huggingface_hub is available: 1.10.1 → 1.11.0

  Do you want to update now? [Y/n] (/usr/bin/python3 -m pip install -U huggingface_hub) 
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help

^C
Traceback (most recent call last):
  File "/content/kikuyu-tts/scripts/colab_train_vits_scratch.py", line 157, in <module>
    main()
  File "/content/kikuyu-tts/scripts/colab_train_vits_scratch.py", line 131, in main
    resume_ckpt = merge_resume_checkpoint(local_output_dir, drive_output_dir)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/kikuyu-tts/scripts/colab_train_vits_scratch.py", line 31, in merge_resume_checkpoint
    if not drive_output_dir.exists():
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File 

Create a metrics CSV at `artifacts/checkpoint_metrics.csv` with columns:
- `checkpoint`
- `synthesis_success_rate`
- `clipping_rate`
- `mos_lite`
- `wer_proxy`


In [31]:
!python scripts/evaluate_and_select.py \
  --metrics-csv artifacts/checkpoint_metrics.csv \
  --out-json artifacts/best_checkpoint_selection.json \
  --out-csv artifacts/tts_eval_summary.csv


Traceback (most recent call last):
  File "/content/kikuyu-tts/scripts/evaluate_and_select.py", line 81, in <module>
    main()
  File "/content/kikuyu-tts/scripts/evaluate_and_select.py", line 44, in main
    with metrics_path.open("r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 1013, in open
    return io.open(self, mode, buffering, encoding, errors, newline)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/checkpoint_metrics.csv'


In [32]:
!python scripts/prepare_local_integration.py \
  --best-checkpoint-dir artifacts/colab_runs/kikuyu_vits_scratch/checkpoint_best \
  --tokenizer-dir artifacts/tokenizer_kikuyu_char \
  --eval-summary-csv artifacts/tts_eval_summary.csv \
  --out-dir artifacts/local_integration/kikuyu_vits_best \
  --model-id kikuyu-vits-scratch-waxal


Traceback (most recent call last):
  File "/content/kikuyu-tts/scripts/prepare_local_integration.py", line 70, in <module>
    main()
  File "/content/kikuyu-tts/scripts/prepare_local_integration.py", line 37, in main
    raise FileNotFoundError(f"Checkpoint dir not found: {best_ckpt_dir}")
FileNotFoundError: Checkpoint dir not found: /content/kikuyu-tts/artifacts/colab_runs/kikuyu_vits_scratch/checkpoint_best
